# ARC2 length-aware decoding: reserve confirmation
Frozen configuration from the promoted pilot. 32 reserve tasks, two arms, one shared adapter per task. Private research only.


## 1. Freeze the run configuration

In [ ]:
import hashlib, importlib.util, json, os, platform, sys, time
from pathlib import Path

# Fix this configuration before inference. "smoke" runs four hashed tasks.
ARC_TASK_SPLIT = "smoke"  # smoke / development / holdout / full
ARC_PRIMARY_SELECTOR = "baseline"  # baseline / family / anchored_family
assert ARC_TASK_SPLIT in {"smoke","development","holdout","full"}
assert ARC_PRIMARY_SELECTOR in {"baseline","family","anchored_family"}
rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN", "").lower() in {"1","true","yes"}
os.environ["KAGGLE_IS_COMPETITION_RERUN"] = "1" if rerun_mode else ""
os.environ["PYTHONHASHSEED"] = "0"
os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"
os.environ["OMP_NUM_THREADS"] = "12"
if Path("/usr/local/cuda/bin/ptxas").exists():
    os.environ["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"
global_end_time = time.time() + 10800 - 300
artifacts = Path("/kaggle/working/arc_artifacts")
artifacts.mkdir(exist_ok=True)
for i in range(4):
    (artifacts / f"worker{i}").unlink(missing_ok=True)
dependency_paths = [Path("/kaggle/usr/lib/pip_install_unsloth_flash_patch"),
                    Path("/kaggle/input/pip-install-unsloth-flash-patch")]
for p in dependency_paths:
    if (p / "unsloth").is_dir():
        sys.path.insert(0,str(p))
        os.environ["PYTHONPATH"] = str(p) + os.pathsep + os.environ.get("PYTHONPATH", "")
        break
import torch
assert torch.cuda.is_available(), "This baseline needs CUDA; select Kaggle GPU L4 x4."
assert torch.cuda.device_count() >= 4, "Select GPU L4 x4 to reproduce the four-worker baseline."
assert importlib.util.find_spec("unsloth") is not None, "Attach sorokin/pip-install-unsloth-flash-patch outputs or use the original compatible runtime."
model_paths = [Path("/kaggle/input/qwen3_4b_grids15_sft139/transformers/bfloat16/1"),
               Path("/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1")]
model_path = next((p for p in model_paths if (p/"config.json").exists()),None)
assert model_path is not None, "Attach sorokin/qwen3_4b_grids15_sft139, Transformers/bfloat16, version 1."
os.environ["ARC_MODEL_PATH"] = str(model_path)
manifest = {"upstream": "https://www.kaggle.com/code/koushikrudra/failed-in-aimo?scriptVersionId=312063593",
            "upstream_public_score": 33.89, "upstream_version": 1,
            "upstream_notebook_sha256": "71416dd4d9885ecf2ae99b8b72ff0ef66d31fac0354b79d2d11071c49f84111d",
            "split": "competition_hidden" if rerun_mode else ARC_TASK_SPLIT,
            "primary_selector": ARC_PRIMARY_SELECTOR,
            "python": sys.version, "torch": torch.__version__,
            "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
            "started_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ",time.gmtime()),
            "seed_policy": "BLAKE2b(task_id) and PYTHONHASHSEED=0",
            "validation_status": "runtime not yet completed"}
(artifacts / "manifest.json").write_text(json.dumps(manifest,indent=2))
print(json.dumps(manifest,indent=2))


In [ ]:
import subprocess
subprocess.run([sys.executable,"-m","pip","uninstall","-y","tensorflow"],check=False)


## 2. Materialize the pinned modules and explicit patches

In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries={k: {"train": v["train"], "test": [{"input": p["input"]} for p in v["test"]]} for k,v in json.loads(queries).items()},
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

In [ ]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]


class ArcDecoder:
    
    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        for key in sorted(os.listdir(store)):
            with bz2.BZ2File(os.path.join(store, key)) as f:
                outputs = pickle.load(f)
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
        print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")

In [ ]:
%%writefile fast_dfs.py
"""Iterative threshold DFS that reproduces the parent ``turbo_dfs`` search.

The parent Notebook (``notebooks/baseline_33_89_source.py``, lines 506-595)
explores continuations recursively: one Python frame per generated token, a
full-vocabulary ``log_softmax`` copied to the CPU at every step, and one
``.item()`` call per ARC token and row.  This module keeps the same search
semantics and changes only how the work is executed:

* an explicit stack replaces recursion, so answer length is not bounded by the
  interpreter recursion limit (a 30x30 grid needs 931 decoding levels);
* the log-softmax is computed where the logits live and only the ARC-token
  columns (16 of ~152k) are transferred to the host;
* the per-row candidate lists are built from one ``tolist()`` call.

Search semantics that are preserved exactly (see ``scripts/test_fast_dfs.py``):

* rows of a batch advance in lockstep; at every level each row pops its next
  candidate, exhausted rows are fed ``pad_id`` with score 1000;
* children are visited in stable ascending score order, ties keep the order
  of ``arc_tokens``;
* an end token within the cutoff is recorded as a finished suffix at the level
  where it appears; non-end tokens are only expanded while
  ``max_new_tokens > 1``;
* the deadline test ``now - start < time_cap and now < end_time`` is applied
  at the top of every level loop, so a deadline unwinds the stack in the same
  order as the recursive version;
* the cache returned by the forward call at depth ``d`` is the cache passed to
  every child of depth ``d``.  With the Unsloth inference path the length of
  the passed cache fixes the write position, which is what makes sibling
  branches overwrite each other correctly.  ``prepare_cache`` lets a caller
  enforce the same semantics on an in-place cache (for example by cropping a
  ``transformers`` ``DynamicCache`` to ``pos``).

When ``device_log_softmax`` is true the log-softmax is evaluated on the model
device.  On CUDA the reduction order differs from the CPU reduction used by the
parent, so scores may differ in the last float32 bits; set it to false to keep
the parent arithmetic bit-for-bit.
"""
from __future__ import annotations

import time
from collections import defaultdict
from typing import Callable, Optional

import torch

DEAD_SCORE = 1000.0


class _Frame:
    __slots__ = ("pos", "cache", "remaining", "candidates", "suffixes", "child_tokens")

    def __init__(self, pos, cache, remaining, candidates, suffixes):
        self.pos = pos
        self.cache = cache
        self.remaining = remaining          # max_new_tokens at this level
        self.candidates = candidates        # list[list[(score, token)]] per row
        self.suffixes = suffixes            # defaultdict(list) per row
        self.child_tokens = None            # tokens fed to the active child


def _arc_log_probs(logits: torch.Tensor, arc_index: torch.Tensor, device_log_softmax: bool) -> torch.Tensor:
    """Return float32 CPU log probabilities of the ARC-token columns, shape (n, |arc|)."""
    if device_log_softmax:
        logp = logits.float().log_softmax(-1).index_select(-1, arc_index.to(logits.device))
        return logp.cpu()
    return logits.float().cpu().log_softmax(-1).index_select(-1, arc_index)


def _expand(logits, scores, remaining, max_score, arc_tokens, arc_index, eos_id,
            device_log_softmax, trace):
    """Build the candidate lists and finished suffixes of one level."""
    if trace is not None:
        trace["prefix_calls"] = trace.get("prefix_calls", 0) + 1
    logp = _arc_log_probs(logits, arc_index, device_log_softmax)
    n = logp.size(0)
    # Same float32 operation as the parent: tensor(scores).view(n, 1) - log_softmax.
    rows = (torch.tensor(scores, dtype=torch.float32).view(n, 1) - logp).tolist()
    candidates, suffixes = [], defaultdict(list)
    for i, row in enumerate(rows):
        found = []
        for t, score in zip(arc_tokens, row):
            if score < max_score:
                if t == eos_id:
                    if trace is not None:
                        trace["eos_within_cutoff"] = trace.get("eos_within_cutoff", 0) + 1
                    suffixes[i].append((score, [t]))
                elif remaining > 1:
                    found.append((score, t))
            elif trace is not None:
                trace["pruned_extensions"] = trace.get("pruned_extensions", 0) + 1
        found.sort(key=lambda x: x[0])
        candidates.append(found)
    return candidates, suffixes


def _merge_into_parent(child: _Frame, parent: _Frame) -> None:
    for batch_id, beams in child.suffixes.items():
        head = parent.child_tokens[batch_id]
        for score, tokens in beams:
            parent.suffixes[batch_id].append((score, [head] + tokens))


def turbo_dfs_iterative(model, logits, max_new_tokens, max_score, scores, pos, cache,
                        start_time, end_time, *, arc_tokens, eos_id, pad_id,
                        time_cap: float = 540.0, device_log_softmax: bool = True,
                        prepare_cache: Optional[Callable] = None, trace: Optional[dict] = None,
                        clock: Callable[[], float] = time.time,
                        extend_until: Optional[float] = None) -> dict:
    """Drop-in iterative replacement for the parent ``turbo_dfs``.

    Returns ``{row: [(score, tokens), ...]}`` with the same content and append
    order as the recursive version.

    ``extend_until`` (default off) enables first-descent extension: while the
    batch is still on its first greedy descent and no row has finished a
    candidate, the deadline is ``extend_until`` instead of ``end_time``.  The
    extension ends as soon as any suffix is recorded, any level starts its
    second child, or every row has been pruned.  With ``extend_until=None``
    the search is identical to the parent.
    """
    arc_tokens = list(arc_tokens)
    arc_index = torch.tensor(arc_tokens, dtype=torch.long)
    n = logits.size(0)

    def expand(level_logits, level_scores, remaining):
        return _expand(level_logits, level_scores, remaining, max_score, arc_tokens, arc_index,
                       eos_id, device_log_softmax, trace)

    cands, sufs = expand(logits, list(scores), max_new_tokens)
    root = _Frame(pos, cache, max_new_tokens, cands, sufs)
    stack = [root]
    device = getattr(model, "device", logits.device)
    extending = extend_until is not None and not sufs

    while stack:
        frame = stack[-1]
        deadline = extend_until if extending else end_time
        # Two clock reads with short-circuit, exactly as in the parent loop condition.
        if not (clock() - start_time < time_cap and clock() < deadline):
            stack.pop()
            if stack:
                _merge_into_parent(frame, stack[-1])
            continue

        if extending and frame.child_tokens is not None:
            extending = False                     # this level would start its second child
            if not clock() < end_time:
                continue

        batch_tokens, batch_scores, alive = [], [], 0
        for i in range(n):
            if frame.candidates[i]:
                score, t = frame.candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                alive += 1
            else:
                batch_tokens.append(pad_id)
                batch_scores.append(DEAD_SCORE)
        if alive == 0:
            extending = False
            stack.pop()
            if stack:
                _merge_into_parent(frame, stack[-1])
            continue

        level_cache = frame.cache
        if prepare_cache is not None:
            level_cache = prepare_cache(level_cache, frame.pos)
        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), frame.pos, device=device),
            past_key_values=level_cache,
            return_dict=True,
            use_cache=True,
        )
        frame.child_tokens = batch_tokens
        cands, sufs = expand(outputs.logits[:, -1], batch_scores, frame.remaining - 1)
        if sufs:
            extending = False
        stack.append(_Frame(frame.pos + 1, outputs.past_key_values, frame.remaining - 1, cands, sufs))

    if trace is not None and clock() >= end_time:
        trace["deadline_hit"] = True
    return root.suffixes


@torch.no_grad()
def inference_turbo_dfs_iterative(model, prefix_tokens, max_new_tokens, max_score, end_time, *,
                                  arc_tokens, eos_id, pad_id, time_cap: float = 540.0,
                                  device_log_softmax: bool = True, prepare_cache=None,
                                  trace=None, clock=time.time, extend_until=None):
    """Drop-in replacement for the parent ``inference_turbo_dfs``."""
    device = getattr(model, "device", None)
    input_ids = torch.tensor(prefix_tokens, device=device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs_iterative(
        model, outputs.logits[:, -1], max_new_tokens, max_score, [0.0] * input_ids.size(0),
        input_ids.size(1), outputs.past_key_values, clock(), end_time,
        arc_tokens=arc_tokens, eos_id=eos_id, pad_id=pad_id, time_cap=time_cap,
        device_log_softmax=device_log_softmax, prepare_cache=prepare_cache, trace=trace, clock=clock,
        extend_until=extend_until)
    return [(batch_id, sorted(beams, key=lambda x: x[0])) for batch_id, beams in suffixes.items()]


In [ ]:
%%writefile arc_solver.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc
import json
import hashlib
from fast_dfs import inference_turbo_dfs_iterative
from neural_evidence import stable_seed
import os
import io
import time
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle

logging.disable(logging.WARNING)

ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15


class UnslothFixedTrainer(UnslothTrainer):

    # Issue https://github.com/unslothai/unsloth/issues/2435

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        # 🔧 KEY FIX: Clone the loss tensor before in-place operations
        if hasattr(loss, "clone"):
            loss = loss.clone()  # Converts view tensor to independent tensor
        # Now safe for DDP gradient scaling
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time, trace=None) -> dict:

    if trace is not None: trace["prefix_calls"] = trace.get("prefix_calls", 0) + 1
    n = logits.size(0)

    nll = torch.tensor(scores, dtype=torch.float32).view(n, 1) - logits.float().cpu().log_softmax(-1)

    suffixes = defaultdict(list)

    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for t in ARC_TOKENS:
            score = nll[i, t].item()
            if trace is not None and score >= max_score:
                trace["pruned_extensions"] = trace.get("pruned_extensions", 0) + 1
            if score < max_score:
                if t == EOS_ID:
                    if trace is not None: trace["eos_within_cutoff"] = trace.get("eos_within_cutoff", 0) + 1
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0]) #[:5]
    
    while time.time() - start_time < 540 and time.time() < end_time:

        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
        trace=trace,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    if trace is not None and time.time() >= end_time: trace["deadline_hit"] = True
    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time, trace=None):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
        trace=trace,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    batch_logits = outputs.logits.float().cpu().log_softmax(-1)
    result = []
    for logits, query_tokens, answer_tokens in zip(batch_logits, batch_query_tokens, batch_answer_tokens):
        query_length = len(query_tokens)
        answer_logits = logits[query_length-1:query_length-1+len(answer_tokens)]
        answer_score = answer_logits[torch.arange(len(answer_tokens)), answer_tokens].sum()
        result.append(-answer_score.item())
    return result


def worker(rank, queue, end_time):

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        # Disable FSDP (use standard DDP)
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=os.environ["ARC_MODEL_PATH"],
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()

    max_score = -np.log(0.2)

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    arc_test_set = ArcDataset.from_file(test_path)

    dir_outputs = "/kaggle/working/arc_artifacts/inference_outputs"
    os.makedirs(dir_outputs, exist_ok=True)

    while not queue.empty():

        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        key = queue.get()
        if key is None:
            break
        
        start_time = time.time()
        
        torch.cuda.reset_peak_memory_stats()

        load_result = set_peft_model_state_dict(
            model,
            default_weights.copy(),
            adapter_name="default",
        )

        model = FastLanguageModel.for_training(model)

        puzzle_ds = arc_test_set.change_keys([key])

        train_ds = puzzle_ds.augment(n=16, shfl_keys=True, seed=1)
        train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

        with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            
            trainer = UnslothFixedTrainer(
                model=model,
                tokenizer=tokenizer,
                data_collator=collator,
                train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                dataset_text_field="text",
                max_seq_length=max_seq_length,
                args=UnslothTrainingArguments(**train_args),
            )

            stats = trainer.train()
            assert stats.global_step == 128, "Incomplete shared task adaptation"

            model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)

            del trainer

        model = FastLanguageModel.for_inference(model)
        
        gc.collect()
        torch.cuda.empty_cache()
            
        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for training")

        torch.cuda.reset_peak_memory_stats()
        
        print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

        adapted_hash = hashlib.sha256()
        delta_sq = 0.
        changed_tensors = 0
        for name, tensor in sorted(get_peft_model_state_dict(model, adapter_name="default").items()):
            delta = (tensor.detach().float() - default_weights[name].float())
            delta_sq += float(delta.square().sum().item())
            changed_tensors += int(bool(torch.any(delta != 0).item()))
            adapted_hash.update(name.encode())
            adapted_hash.update(tensor.detach().cpu().contiguous().view(torch.uint8).numpy().tobytes())
        assert changed_tensors > 0 and delta_sq > 0, "Task adaptation did not change parameters"
        puzzle_ds_multi = puzzle_ds.split_multi_replies()

        eval_ds = puzzle_ds_multi.augment(n=2, seed=2)
        eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

        test_id_to_subkeys = defaultdict(list)
        for subkey in sorted(eval_ds.keys):
            test_id = subkey.split(".")[0].split("_")[1]
            test_id_to_subkeys[test_id].append(subkey)

        batches = []
        for test_id, subkeys in test_id_to_subkeys.items():
            # 0: permute x 2
            # 4: rot90.rot90.permute x 2
            batch = []
            for offset in [0, 4]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 2: permute.rot90 x 2
            # 6: rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [2, 6]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
        for test_id, subkeys in test_id_to_subkeys.items():
            # 8: transpose.permute x 2
            # 12: transpose.rot90.rot90.permute x 2
            batch = []
            for offset in [8, 12]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 10: transpose.rot90.permute x 2
            # 14: transpose.rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [10, 14]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)

        task_record = {"task_id": key, "train_seconds": stats.metrics.get("train_runtime"),
                       "training_steps": stats.global_step, "arms": {}, "shared_adaptation": True,
                       "adapted_weights_sha256": adapted_hash.hexdigest(),
                       "adapter_delta_l2": delta_sq ** .5, "changed_adapter_tensors": changed_tensors}
        import itertools
        arm_names = ["control", "length_aware"] if stable_seed(key) % 2 == 0 else ["length_aware", "control"]
        task_record["arm_order"] = arm_names
        for arm_name in arm_names:
            arm_start = time.time()
            arm_deadline = min(arm_start + 180, end_time)
            arm_max_score = -np.log(0.2)
            dir_outputs = f"/kaggle/working/arc_artifacts/{arm_name}/inference_outputs"
            os.makedirs(dir_outputs, exist_ok=True)
            batch_times = []
            print(f"[Rank {rank}] arm {arm_name} start {key}", flush=True)
            with torch.inference_mode():
                
                known_scores = {}
                known_token_counts = {}

                ledger = None
                output_of = [sk[0].split(".")[0] for sk in batches]
                if arm_name == "length_aware":
                    # Each test output keeps an equal share of the arm budget; time saved or
                    # overrun by one output's batches is charged only to that output.
                    outputs_here = sorted(set(output_of))
                    ledger = {o: (arm_deadline - time.time()) / len(outputs_here) for o in outputs_here}
                    left = {o: output_of.count(o) for o in outputs_here}
                for batch_index, subkeys in enumerate(batches):

                    now = time.time()
                    if now >= arm_deadline - 2:
                        break
                    extend_until = None
                    if ledger is None:
                        # Every remaining view batch receives a share of the remaining time.
                        batch_deadline = min(arm_deadline, now + (arm_deadline-now)/(len(batches)-batch_index))
                    else:
                        out_key = output_of[batch_index]
                        if ledger[out_key] <= 2:
                            left[out_key] -= 1
                            batch_times.append({"batch": batch_index, "seconds": 0.0, "view_keys": subkeys,
                                                "skipped_exhausted_output_budget": True})
                            continue
                        batch_deadline = min(arm_deadline, now + ledger[out_key] / left[out_key])
                        extend_until = min(arm_deadline, now + ledger[out_key])
                    batch_started = now

                    print(f"[Rank {rank}] decoding {subkeys}")

                    tokens = []
                    for subkey in subkeys:
                        data = eval_ds.get(subkey, formatter)
                        tokens.append(tokenizer.encode(data["input"]))

                    dfs_trace = {}
                    if arm_name == "control":
                        dfs_result = inference_turbo_dfs(model, tokens, max_new_tokens, arm_max_score,
                                                         batch_deadline, trace=dfs_trace)
                    else:
                        dfs_result = inference_turbo_dfs_iterative(
                            model, tokens, max_new_tokens, arm_max_score, batch_deadline,
                            arc_tokens=ARC_TOKENS, eos_id=EOS_ID, pad_id=PAD_ID, trace=dfs_trace,
                            extend_until=extend_until)
                    dfs_finished = time.time()
                    raw_finished = sum(len(beams) for _, beams in dfs_result)
                    valid_finished = sum(formatter.convert_tokens_to_array(ys) is not None
                                         for _, beams in dfs_result for _, ys in beams)
                    max_suffix_tokens = max([len(ys) for _, beams in dfs_result for _, ys in beams] or [0])


                    for subkey_id, scored_beams in dfs_result:

                        subkey = subkeys[subkey_id]
                        bk = subkey.split(".")[0]
                        decoded_result = []

                        for beam_score, tokens in scored_beams:

                            array = formatter.convert_tokens_to_array(tokens)
                            if array is None:
                                continue

                            solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)

                            grid_id = (bk, tuple(map(tuple, solution)))

                            if grid_id in known_scores:
                                augmented_scores = known_scores[grid_id]
                                augmented_token_counts = known_token_counts[grid_id]
                            else:
                                print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                                aug_dataset = ArcDataset(
                                    keys=[bk],
                                    queries={bk: puzzle_ds_multi.queries.get(bk)},
                                    replies={bk: [solution.tolist()]},
                                )
                                aug_dataset = aug_dataset.augment(seed=stable_seed(bk))
                                aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                                aug_queries = []
                                aug_answers = []
                                for augmented_sample in aug_dataset.as_list(formatter):
                                    aug_queries.append(augmented_sample["input"])
                                    aug_answers.append(augmented_sample["reply"])
                                augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                                augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                                augmented_scores = augmented_scores1 + augmented_scores2
                                known_scores[grid_id] = augmented_scores
                                augmented_token_counts = [len(tokenizer.encode(a)) for a in aug_answers]
                                known_token_counts[grid_id] = augmented_token_counts
                        
                            decoded_result.append({
                                "beam_score": beam_score,
                                "score_aug": augmented_scores,
                                "score_aug_tokens": augmented_token_counts,
                                "solution": solution,
                            })

                        if len(decoded_result):
                            with bz2.BZ2File(os.path.join(dir_outputs, subkey), "w") as f:
                                pickle.dump(decoded_result, f)

                    batch_times.append({"batch": batch_index, "seconds": time.time()-batch_started,
                                        "view_keys": subkeys, "dfs_trace": dfs_trace,
                                        "dfs_seconds": dfs_finished-batch_started,
                                        "slice_seconds": batch_deadline-batch_started,
                                        "extension_cap_seconds": None if extend_until is None else extend_until-batch_started,
                                        "overran_slice": dfs_finished > batch_deadline + 0.5,
                                        "raw_finished": raw_finished, "valid_finished": valid_finished,
                                        "max_suffix_tokens": max_suffix_tokens})
                    if ledger is not None:
                        ledger[out_key] -= time.time() - batch_started
                        left[out_key] -= 1

            task_record["arms"][arm_name] = {
                "seconds": time.time()-arm_start, "requested_seconds": 180,
                "batch_count": len(batch_times), "total_batches": len(batches),
                "batches": batch_times, "sequence_probability_floor": float(np.exp(-arm_max_score)),
                "global_budget_truncated": arm_start+180 > end_time}
            print(f"[Rank {rank}] arm {arm_name} finished {key}: {task_record['arms'][arm_name]['seconds']:.1f}s", flush=True)
        telemetry_dir = "/kaggle/working/arc_artifacts/task_telemetry"
        os.makedirs(telemetry_dir, exist_ok=True)
        with open(os.path.join(telemetry_dir, key+".json"), "w") as telemetry_file:
            json.dump(task_record, telemetry_file, indent=2)

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")
        
        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")

In [ ]:
%%writefile starter.py
import os
import time
import json
import torch
import argparse
import torch.multiprocessing as mp


def local_worker(rank, queue, end_time):
    
    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)

    torch.set_default_device("cpu")

    # Fix Unsloth patching issue
    if rank > 0:
        while not os.path.exists(f"/kaggle/working/arc_artifacts/worker{rank-1}"):
            time.sleep(5)
    
    from arc_solver import worker

    with open(f"/kaggle/working/arc_artifacts/worker{rank}", "w") as f:
        f.write("Ok")
    
    print(f"[Rank {rank}] start!")
    
    worker(rank, queue, end_time)
    
    print(f"[Rank {rank}] done!")


if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    args = parser.parse_args()

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    with open(test_path, "r") as f:
        data = json.load(f)

    queue = mp.Manager().Queue()

    selected_path = os.environ.get("ARC_TASK_IDS_FILE")
    selected = set(json.load(open(selected_path))) if selected_path and not rerun_mode else set(data)
    for key in sorted(data):
        if key in selected:
            queue.put(key)
    for _ in range(4):
        queue.put(None)
    
    mp.spawn(local_worker, args=(queue, args.end_time), nprocs=4)

In [ ]:
%%writefile neural_evidence.py
"""Input-only replay selectors for the pinned public Qwen/NVARC candidate pool.

MIT-0. Original code. The baseline scores are reimplemented for comparison;
the upstream notebook retains its Apache-2.0 license and attribution.
"""
from __future__ import annotations
from collections import defaultdict
import hashlib
import math
import statistics


def stable_seed(task_id, modulus=1024**2):
    return int.from_bytes(hashlib.blake2b(str(task_id).encode(),digest_size=8).digest(),'little') % modulus


def grid_tuple(grid):
    if hasattr(grid,'tolist'):grid=grid.tolist()
    if not isinstance(grid,(list,tuple)) or not 1<=len(grid)<=30:
        raise ValueError('grid must have 1..30 rows')
    width=len(grid[0]) if isinstance(grid[0],(list,tuple)) else 0
    if not 1<=width<=30:raise ValueError('grid must have 1..30 columns')
    result=[]
    for row in grid:
        if not isinstance(row,(list,tuple)) or len(row)!=width:raise ValueError('ragged grid')
        if any(isinstance(v,bool) or not isinstance(v,int) or v<0 or v>9 for v in row):
            raise ValueError('grid colors must be integers 0..9')
        result.append(tuple(row))
    return tuple(result)


def geometric_family(source_key):
    """Canonical D4 action, independent of color permutation and beam index."""
    a=((0,1,2),(3,4,5),(6,7,8))
    for op in source_key.split('.')[1:]:
        if op=='rot90':a=tuple(zip(*a))[::-1]
        elif op=='transpose':a=tuple(zip(*a))
        elif op.startswith(('permute','copy','out','ex','run')):continue
        else:raise ValueError(f'unknown augmentation descriptor: {op}')
    return tuple(v for row in a for v in row)


def clean_records(guesses):
    result=[]
    for source_key,g in guesses.items():
        try:
            y=grid_tuple(g['solution'])
            aug=tuple(float(x) for x in g['score_aug'])
            beam=float(g['beam_score'])
            if not aug or not all(math.isfinite(x) for x in aug+(beam,)):continue
            tokens=g.get('score_aug_tokens')
            if tokens is not None:
                if len(tokens)!=len(aug) or any(float(x)<=0 for x in tokens):continue
                normalized=statistics.fmean(x/float(n) for x,n in zip(aug,tokens))
            else:
                # For old traces, leave summed NLL unchanged. The field flags it.
                normalized=statistics.fmean(aug)
            family=geometric_family(source_key)
            result.append({'source':source_key,'grid':y,'family':family,'beam':beam,
                           'aug_mean':statistics.fmean(aug),'normalized_nll':normalized,
                           'has_token_counts':tokens is not None})
        except (KeyError,TypeError,ValueError,OverflowError):continue
    return result


def baseline_rank(guesses):
    """Original score_kgmon: number of beams minus mean augmented summed NLL."""
    groups=defaultdict(list)
    for r in clean_records(guesses):groups[r['grid']].append(r)
    scores={y:len(rs)-statistics.fmean(r['aug_mean'] for r in rs) for y,rs in groups.items()}
    return sorted(groups,key=lambda y:(-scores[y],y))


def family_rank(guesses):
    """Each geometric view supports an output once, regardless of beam clones.

    This is a fixed hypothesis about useful evidence, not a fitted probability.
    View support first; per-token likelihood breaks ties. Neither signal uses
    the hidden target. Rotated views are correlated, not independent samples.
    """
    groups=defaultdict(lambda:defaultdict(list))
    for r in clean_records(guesses):groups[r['grid']][r['family']].append(r['normalized_nll'])
    diagnostics=[]
    for y,by_family in groups.items():
        per_view=[min(xs) for xs in by_family.values()]
        diagnostics.append({'grid':y,'family_support':len(per_view),
                            'mean_per_view_nll':statistics.fmean(per_view)})
    return sorted(diagnostics,key=lambda r:(-r['family_support'],r['mean_per_view_nll'],r['grid']))


def anchored_family_select(guesses, baseline_outputs=None):
    """Keep baseline attempt 1; replace attempt 2 only with >=2-view evidence.

    It can improve or reduce pass@2; no monotonic-accuracy guarantee is made.
    The complete family ranking is clone-invariant; the anchored final selector
    inherits any clone sensitivity of the baseline anchor.
    """
    baseline=(baseline_rank(guesses) if baseline_outputs is None
              else [grid_tuple(y) for y in baseline_outputs])
    if not baseline:return []
    anchor=baseline[0]
    eligible=[r for r in family_rank(guesses) if r['grid']!=anchor and r['family_support']>=2]
    second=eligible[0]['grid'] if eligible else baseline[min(1,len(baseline)-1)]
    return [anchor,second]


def family_select(guesses):
    return [r['grid'] for r in family_rank(guesses)]


def validate_submission(challenges,submission):
    if set(submission)!=set(challenges):raise ValueError('task IDs do not match challenges')
    outputs=0
    for tid,t in challenges.items():
        rows=submission[tid]
        if len(rows)!=len(t['test']):raise ValueError(f'{tid}: test count mismatch')
        for row in rows:
            if set(row)!={'attempt_1','attempt_2'}:raise ValueError(f'{tid}: expected exactly two attempts')
            grid_tuple(row['attempt_1']);grid_tuple(row['attempt_2']);outputs+=1
    return {'tasks':len(challenges),'test_outputs':outputs,'valid':True}


def score_submissions(solutions,submissions):
    """Post-hoc scoring; this function is never called by a selector."""
    result={}
    for name,submission in submissions.items():
        rows=[]
        for tid,ys in solutions.items():
            for i,y in enumerate(ys):
                truth=grid_tuple(y);p=submission[tid][i]
                rows.append({'task_id':tid,'test_index':i,
                             'pass1':int(grid_tuple(p['attempt_1'])==truth),
                             'pass2':int(truth in (grid_tuple(p['attempt_1']),grid_tuple(p['attempt_2'])))})
        result[name]={'test_outputs':len(rows),'pass1':sum(r['pass1'] for r in rows)/len(rows),
                      'pass2':sum(r['pass2'] for r in rows)/len(rows),'per_output':rows}
    return result


## 3. Select task IDs without reading target outputs

In [ ]:
from arc_loader import ArcDataset
assert not rerun_mode, "This notebook is a public-development experiment only."
roots = [Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-2"),
         Path("/kaggle/input/arc-prize-2026-arc-agi-2")]
competition_root = next(p for p in roots if (p/"arc-agi_evaluation_challenges.json").exists())
assert competition_root == roots[0]
challenges_path = competition_root/"arc-agi_evaluation_challenges.json"
assert hashlib.sha256(challenges_path.read_bytes()).hexdigest() == "e7c62a4bd211867c6b538f66b8013b81f299663c82ca062f49a52bf439d6e4e8"
full_data = ArcDataset.from_file(str(challenges_path))
ordered = sorted(full_data.keys, key=lambda k: hashlib.sha256(("arc-neural-v1:"+k).encode()).hexdigest())
selected_ids = ['f560132c', '221dfab4', '4a21e3da', 'cbebaa4b', '135a2760', '7b0280bc', 'd59b0160', '247ef758', 'edb79dae', 'b9e38dc0', 'faa9f03d', '2d0172a1', 'a47bf94d', '8e5c0c38', 'bf45cf4b', 'f931b4a8', '2b83f449', '20a9e565', 'dd6b8c4b', '71e489b6', '981571dc', '7c66cb00', '3a25b0d8', '58f5dbd5', '88bcf3b4', '97d7923e', '53fb4810', '3e6067c3', '800d221b', '78332cb0', '7491f3cf', 'c7f57c3e']
assert len(selected_ids) == 32 and set(selected_ids) <= set(ordered)
ids_path = artifacts/"selected_task_ids.json"
ids_path.write_text(json.dumps(selected_ids, indent=2))
os.environ["ARC_TASK_IDS_FILE"] = str(ids_path)
data = full_data.change_keys(selected_ids, keep_flags=True)
manifest.update({"split": "reserve_confirmation_32", "task_count": len(selected_ids),
                 "test_output_count": sum(len(t["test"]) for t in data.queries.values()),
                 "challenge_sha256": hashlib.sha256(challenges_path.read_bytes()).hexdigest(),
                 "selected_task_ids_sha256": hashlib.sha256(ids_path.read_bytes()).hexdigest(),
                 "arm_seconds": 180, "arms": {"control": 0.2, "length_aware": 0.2},
                 "adaptation": "shared task-specific model, 128 augmented samples; arm order counterbalanced by task hash",
                 "scheduling": "same remaining-time-per-view-batch policy in both arms",
                 "notebook_timeout_seconds": 10800})
for arm in ["control", "length_aware"]:
    (artifacts/arm/"inference_outputs").mkdir(parents=True, exist_ok=True)
(artifacts/"manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2), flush=True)

protocol = {'tasks': ['f560132c', '221dfab4', '4a21e3da', 'cbebaa4b', '135a2760', '7b0280bc', 'd59b0160', '247ef758', 'edb79dae', 'b9e38dc0', 'faa9f03d', '2d0172a1', 'a47bf94d', '8e5c0c38', 'bf45cf4b', 'f931b4a8', '2b83f449', '20a9e565', 'dd6b8c4b', '71e489b6', '981571dc', '7c66cb00', '3a25b0d8', '58f5dbd5', '88bcf3b4', '97d7923e', '53fb4810', '3e6067c3', '800d221b', '78332cb0', '7491f3cf', 'c7f57c3e'], 'test_outputs': 48, 'scope': 'Fresh confirmation on the 32 reserve tasks; first use of these tasks', 'arms': {'control': 'parent recursive DFS, equal remaining-time slices', 'length_aware': 'iterative DFS, per-output ledger, first-descent extension'}, 'seconds_per_arm': 180, 'cutoff_probability': 0.2, 'frozen_from': 'length_aware_development (pilot decision promote_to_fresh_confirmation)', 'primary_comparison': 'length_aware minus control pass@2, paired by output, task bootstrap', 'secondary': ['empty valid pools', 'candidate oracle', 'gains and losses', 'time per forward pass'], 'stop_rule': 'Report whatever is observed; no further tuning on these tasks', 'competition_submission': False, 'reserve_tasks_consumed': 32, 'budget': {'notebook_wall_seconds': 10800, 'max_quota_hours': 2.0}, 'launch_authorization': 'User on 2026-11-04 13:15 and 19:22 (+08:00): pursue the paper prize by any method and continue', 'fast_dfs_sha256': '1803a077719ae495a6e8a07591dff4d9529116518cd3de16ae7c42af5d49efea', 'runtime_sha256': '0b11c57c9beced50e3ac270d5575859ed8684ebbc29f3eb7d6a4a24f98bb218c'}
(artifacts/"protocol.json").write_text(json.dumps(protocol,indent=2))


## 4. Run the shared candidate generator

In [ ]:
import subprocess
command = [sys.executable,"starter.py","--end-time",str(global_end_time)]
with (artifacts/"inference.log").open("w") as logfile:
    run = subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,env=os.environ.copy())
    for line in run.stdout:
        logfile.write(line); logfile.flush()
        if any(term in line for term in ("finished", "timeout", "Traceback", "Error", "start!", "done!")):
            print(line.rstrip(),flush=True)
    return_code = run.wait()
assert return_code==0, f"Inference failed ({return_code}); inspect arc_artifacts/inference.log."


## 5. Freeze all selectors, validate two attempts, and save evidence

In [ ]:
import gzip
import numpy as np
from arc_decoder import ArcDecoder, score_kgmon
from neural_evidence import family_select, anchored_family_select, validate_submission
def anchored_with_exact_parent(guesses):
    return anchored_family_select(guesses, baseline_outputs=score_kgmon(guesses))
submissions = {}
for arm in ["control", "length_aware"]:
    arm_dir = artifacts/arm
    decoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)
    decoder.load_decoded_results(str(arm_dir/"inference_outputs"))
    for selector_name, selector in {"baseline": None, "family": family_select,
                                    "anchored_family": anchored_with_exact_parent}.items():
        selected = decoder.run_selection_algo() if selector is None else decoder.run_selection_algo(selector)
        selected = {k: [np.asarray(y, dtype=int) for y in ys] for k, ys in selected.items()}
        submission = data.get_submission(selected)
        validate_submission(data.queries, submission)
        submissions[arm+"_"+selector_name] = submission
        (arm_dir/f"submission_{selector_name}.json").write_text(json.dumps(submission))
    with gzip.open(arm_dir/"candidate_trace.jsonl.gz", "wt") as f:
        for task_output, guesses in sorted(decoder.decoded_results.items()):
            candidates = []
            for source_key, g in sorted(guesses.items()):
                item = {k: (v.tolist() if hasattr(v, "tolist") else v) for k, v in g.items()}
                item["source_key"] = source_key
                candidates.append(item)
            f.write(json.dumps({"task_output": task_output, "candidates": candidates})+"\n")
Path("submission.json").write_text(json.dumps(submissions["control_baseline"]))
telemetry = [json.loads(p.read_text()) for p in sorted((artifacts/"task_telemetry").glob("*.json"))]
manifest["completed_task_ids"] = [t["task_id"] for t in telemetry]
manifest["all_selected_tasks_completed"] = set(manifest["completed_task_ids"]) == set(selected_ids)
manifest["completed_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
manifest["validation_status"] = "predictions frozen; six submission schemas valid (two arms)"
(artifacts/"manifest.json").write_text(json.dumps(manifest, indent=2))
assert manifest["all_selected_tasks_completed"], "Incomplete pilot: inspect per-task telemetry before interpreting accuracy."


## 6. Score the frozen predictions

In [ ]:
# Labels enter only after both arms and all six submissions are frozen.
from neural_evidence import score_submissions
all_solutions = json.loads((competition_root/"arc-agi_evaluation_solutions.json").read_text())
solutions = {k: all_solutions[k] for k in selected_ids}
metrics = score_submissions(solutions, submissions)
(artifacts/"posthoc_metrics.json").write_text(json.dumps(metrics, indent=2))
print(json.dumps({k: {m: v for m, v in r.items() if m != "per_output"} for k, r in metrics.items()}, indent=2))
print("Thirty-two-task reserve confirmation; leaderboard accuracy requires a separate competition run.")
